In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import seaborn as sns
import plotly.graph_objects as go
from plotly.offline import iplot
from statsmodels.tsa.stattools import adfuller
from tools import utils

ModuleNotFoundError: No module named 'utils'

In [ ]:
train = None
with open("../../Data/PHM2025_training_data/training_data.csv", "r") as f:
    train = pd.read_csv(f)

train.head()

In [ ]:
# Conteggio numero di righe totali e per motore
rows = len(train)
print(f"Numero totale di righe: {rows}")
print(f"\nNumero di righe per ogni motore:")
train.groupby('ESN').size()

In [ ]:
# Conteggio dei valori nulli per colonna
train.isnull().sum()

# Colonne con valori nulli:
# - Sensed_WFuel : peso misurato del carburante
# - Sensed_Core_Speed : velocità di rotazione misurata dello 
#   spool ad alta pressione, che comprende l'albero con il HPC e la HPT
# - Sensed_T3 : temperatura dell'aria misurata in uscita al HPC
# - Sensed_Ps3 : pressione misurata in uscita al HPC
# - Sensed_T45 : temperatura del gas combusto misurata all'uscita della HPT
# - Sensed_T5 : temperatura misurata all'uscita della turbina a bassa pressione

In [ ]:
# Conteggio dei valori nulli per ogni motore
train.groupby('ESN').apply(lambda x: x.isnull().sum(), include_groups=False)

# Da valutare 
# Sono presenti maggiormente nel motore 104

In [ ]:
# Add a cycle index 'scaled_index' for each ESN
train['scaled_index'] = train.groupby('ESN').cumcount()

In [ ]:
# Identificazione degli event points
# Assuming your DataFrame is 'df' and the column is 'sensor_value'
wws_points = train[train['Cumulative_WWs'] > train['Cumulative_WWs'].shift(1)]
hpc_points = train[train['Cumulative_HPC_SVs'] > train['Cumulative_HPC_SVs'].shift(1)]
hpt_points = train[train['Cumulative_HPT_SVs'] > train['Cumulative_HPT_SVs'].shift(1)]

In [1]:
# Calcolo della media e della deviazione standard tra i cicli tra ogni ww per motore
wws_points_2 = wws_points.copy() # lavoro su una copia
wws_counts = wws_points_2.groupby('ESN').size().reset_index(name='Totale_Eventi_WW')
wws_points_2['cycles_between_ww'] = wws_points_2.groupby('ESN')['Cycles_Since_New'].diff()
ww_stats = wws_points_2.groupby('ESN').agg(
    Media_Intervallo_WW=('cycles_between_ww', 'mean'),
    DevStd_Intervallo_WW=('cycles_between_ww', 'std'),
    Num_Eventi=('ESN', 'count')
).reset_index()
print(ww_stats)
# Grafico per chiarezza
ww_stats['ESN'] = ww_stats['ESN'].astype(str)
fig_ww = px.bar(
    ww_stats, 
    x='ESN', 
    y='Media_Intervallo_WW',
    error_y='DevStd_Intervallo_WW',
    text='Num_Eventi',
    title='Media Cicli tra Water Wash (WW) per Motore',
    labels={
        'ESN': 'Motore (ESN)', 
        'Media_Intervallo_WW': 'Media Cicli tra WW',
        'Num_Eventi': 'Numero di eventi'
    },
    color='Media_Intervallo_WW',
    color_continuous_scale='Viridis'
)
fig_ww.update_traces(
    textposition='inside',    
    insidetextanchor='start'
)
fig_ww.update_layout(
    xaxis_tickangle=-45,
    yaxis_title="Cicli Medi (con Dev. Std.)",
    hovermode="x unified",
    height=500
)

# Calcolo della media e della deviazione standard tra i cicli tra ogni HPC_SV per motore
hpc_points_2 = hpc_points.copy() # lavoro su una copia
hpc_counts = hpc_points_2.groupby('ESN').size().reset_index(name='Totale_Eventi_HPC_SV')
hpc_points_2['cycles_between_hpc_sv'] = hpc_points_2.groupby('ESN')['Cycles_Since_New'].diff()
hpc_stats = hpc_points_2.groupby('ESN').agg(
    Media_Intervallo_HPC_SV=('cycles_between_hpc_sv', 'mean'),
    DevStd_Intervallo_HPC_SV=('cycles_between_hpc_sv', 'std'),
    Num_Eventi=('ESN', 'count')
).reset_index()
print(hpc_stats)
# Grafico per chiarezza
hpc_stats['ESN'] = hpc_stats['ESN'].astype(str)
fig_hpc = px.bar(
    hpc_stats, 
    x='ESN', 
    y='Media_Intervallo_HPC_SV',
    error_y='DevStd_Intervallo_HPC_SV',
    text='Num_Eventi',
    title='Media Cicli tra HPC shop visit per Motore',
    labels={
        'ESN': 'Motore (ESN)', 
        'Media_Intervallo_HCP_SV': 'Media Cicli tra HPC SV',
        'Num_Eventi': 'Numero di eventi'
    },
    color='Media_Intervallo_HPC_SV',
    color_continuous_scale='Viridis'
)
fig_hpc.update_traces(
    textposition='inside',    
    insidetextanchor='start'
)
fig_hpc.update_layout(
    xaxis_tickangle=-45,
    yaxis_title="Cicli Medi (con Dev. Std.)",
    hovermode="x unified",
    height=500
)

# Calcolo della media e della deviazione standard tra i cicli tra ogni HPT_SV per motore
hpt_points_2 = hpt_points.copy() # lavoro su una copia
hpt_counts = hpt_points.groupby('ESN').size().reset_index(name='Totale_Eventi_HPT_SV')
hpt_points_2['cycles_between_hpt_sv'] = hpt_points_2.groupby('ESN')['Cycles_Since_New'].diff()
hpt_stats = hpt_points_2.groupby('ESN').agg(
    Media_Intervallo_HPT_SV=('cycles_between_hpt_sv', 'mean'),
    DevStd_Intervallo_HPT_SV=('cycles_between_hpt_sv', 'std'),
    Num_Eventi=('ESN', 'count')
).reset_index()
print(hpt_stats)
# Grafico per chiarezza
hpt_stats['ESN'] = hpt_stats['ESN'].astype(str)
fig_hpt = px.bar(
    hpt_stats, 
    x='ESN', 
    y='Media_Intervallo_HPT_SV',
    error_y='DevStd_Intervallo_HPT_SV',
    text='Num_Eventi',
    title='Media Cicli tra HPT shop visit per Motore',
    labels={
        'ESN': 'Motore (ESN)', 
        'Media_Intervallo_HCT_SV': 'Media Cicli tra HPT SV',
        'Num_Eventi': 'Numero di eventi'
    },
    color='Media_Intervallo_HPT_SV',
    color_continuous_scale='Viridis'
)
fig_hpt.update_traces(
    textposition='inside',    
    insidetextanchor='start'
)
fig_hpt.update_layout(
    xaxis_tickangle=-45,
    yaxis_title="Cicli Medi (con Dev. Std.)",
    hovermode="x unified",
    height=500
)

# Stampa dei grafici
fig_ww.show()
fig_hpc.show()
fig_hpt.show()

NameError: name 'wws_points' is not defined

In [ ]:
# Test per verificare la stazionarietà dei segnali dei sensori - controllo delle statistiche mobili dei segnali

for esn_id in generals.ESN:
    engine_data = train[train['ESN'] == esn_id].copy()
    fig, axes = plt.subplots(4, 4, figsize=(20, 16))
    axes = axes.flatten()
    print(f"Generazione dashboard per Motore ESN: {esn_id}...")
    for i, sensor in enumerate(generals.SENSORS):
        sensor_name = sensor.value if hasattr(sensor, 'value') else sensor
        ax = axes[i]
        series = engine_data[sensor_name].dropna()
        if len(series) > 16:
            roll_mean = series.rolling(window=16).mean()
            roll_std = series.rolling(window=16).std()
            # Plotting
            ax.plot(series.values, alpha=0.3, label='Raw', color='gray')
            ax.plot(roll_mean.values, label='Media Mobile', color='blue', linewidth=1.5)
            ax.plot(roll_std.values, label='Std Mobile', color='red', linewidth=1)
        else:
            ax.text(0.5, 0.5, 'Dati insufficienti', ha='center', va='center')
        # Formattazione singolo grafico
        ax.set_title(f"{sensor_name}", fontsize=10)
        ax.grid(True, alpha=0.2, linestyle='--')
        ax.tick_params(axis='both', which='major', labelsize=8)
        if i == 0:
            ax.legend(loc='upper left', fontsize='x-small')

    fig.suptitle(f"Analisi Stazionarietà - Motore ESN {esn_id}", fontsize=20, y=1.02)
    plt.tight_layout()
    plt.savefig(f"Analisi_Stazionarieta_ESN_{esn_id}.png", bbox_inches='tight')
    plt.show()

In [ ]:
# Test Augmented Dickey-Fuller

results_list = []
for esn_id in generals.ESN:
    engine_data = train[train['ESN'] == esn_id]
    for sensor in generals.SENSORS:
        sensor_name = sensor.value if hasattr(sensor, 'value') else sensor
        series = engine_data[sensor_name].dropna()
        if series.nunique() <= 1:
            results_list.append({'ESN': esn_id, 'Sensor': sensor_name, 'p-value': np.nan, 'Stazionario': 'Dati Costanti'})
            continue
        try:
            res = adfuller(series)
            p_value = res[1]
            is_stationary = True if p_value <= 0.05 else False
            
            results_list.append({
                'ESN': esn_id, 
                'Sensor': sensor_name, 
                'p-value': round(p_value, 4), 
                'Stazionario': is_stationary
            })
        except Exception as e:
            results_list.append({'ESN': esn_id, 'Sensor': sensor_name, 'p-value': np.nan, 'Stazionario': 'Errore'})
df_results = pd.DataFrame(results_list)

# Visualizzazione solo di quelli non stazionari
print(df_results[df_results['Stazionario'] == False].head(20))

In [ ]:
def plot(df, sensor):
  fig = px.line(
        df,
        x=df['scaled_index'],
        y=sensor,
        color='ESN',
        title=f'Andamento del Sensore {sensor} Suddiviso per ESN',
        labels={'x':'Indice del Dato / Tempo (Sequenza)', 'y':f'Valore di {sensor}'},
        height=800,
        line_group='ESN'
    )

  # Aggiorna il layout per un aspetto migliore (opzionale)
  fig.update_xaxes(rangeslider_visible=True) # Aggiunge uno slider in basso per navigare nel tempo

  # Mostra il grafico interattivo in Colab/Jupyter
  fig.show()

In [ ]:
# Matrice di correlazione delle colonne non divisa per ESN
cm = train.corr()
plt.figure(figsize=(15,15))
sns.heatmap(cm, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix Heatmap')
plt.show()

In [ ]:
# Matrice di correlazione delle colonne divisa per ESN
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 18))
axes = axes.flatten()
for idx, i in  enumerate(generals.ESN):
    cm = train[train["ESN"] == i].corr()
    sns.heatmap(cm, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[idx], cbar=False)
    axes[idx].set_title(f'Correlation Matrix - ESN {i}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Mappatura dei colori per gli eventi di manutenzione
EVENT_COLORS = {
    'Cumulative_WWs': 'red',
    'Cumulative_HPC_SVs': 'blue',
    'Cumulative_HPT_SVs': 'green'
}

def create_figure(df, sensor_name, wws_df, hpc_df, hpt_df, esn):
    """Crea una figura Plotly per il sensore specificato."""
    
    df = df[df["ESN"] == esn].copy()
    df.reset_index(drop=True, inplace=True)  # indice locale 0..N-1
    wws_df = wws_df[wws_df["ESN"] == esn]
    hpc_df = hpc_df[hpc_df["ESN"] == esn]
    hpt_df = hpt_df[hpt_df["ESN"] == esn]
    
    fig = px.line(
        df,
        x=df.index,
        y=sensor_name,
        color='ESN',
        title=f'Andamento del Sensore {sensor_name} Suddiviso per ESN',
        labels={'x':'Indice del Dato / Tempo (Sequenza)', 'y':f'Valore di {sensor_name}'},
        height=800,
        line_group='ESN'
    )
    
    for index, row in wws_df.iterrows():
        fig.add_vline(
            x=row.scaled_index,
            line_width=1, line_dash="dash", line_color=EVENT_COLORS['Cumulative_WWs'],
            name='WWs Event',
            # Aggiunge un'annotazione per identificare il tipo di evento
            annotation_text="WWs", annotation_position="top right",
            annotation_font_color=EVENT_COLORS['Cumulative_WWs']
        )
    
    # Eventi Cumulative_HPC_SVs (Manutenzione HPC)
    for index, row in hpc_df.iterrows():
        fig.add_vline(
            x=row.scaled_index,
            line_width=1, line_dash="dash", line_color=EVENT_COLORS['Cumulative_HPC_SVs'],
            name='HPC_SV Event',
            annotation_text="HPC_SV", annotation_position="top left",
            annotation_font_color=EVENT_COLORS['Cumulative_HPC_SVs']
        )
        
    # Eventi Cumulative_HPT_SVs (Manutenzione HPT)
    for index, row in hpt_df.iterrows():
        fig.add_vline(
            x=row.scaled_index,
            line_width=1, line_dash="dash", line_color=EVENT_COLORS['Cumulative_HPT_SVs'],
            name='HPT_SV Event',
            annotation_text="HPT_SV", annotation_position="bottom right",
            annotation_font_color=EVENT_COLORS['Cumulative_HPT_SVs']
        )


    fig.update_layout(
        title=f'Andamento del sensore "{sensor_name}" vs Cicli con Eventi di Manutenzione',
        xaxis_title='Cicli Dall\'Inizio (Cycles_Since_New)',
        yaxis_title=sensor_name,
        legend_title="ESN (Motore)",
        height=600,
        hovermode="x unified"
    )

    return fig


def update_plot(sensor_name, data, esn):
    return create_figure(data, sensor_name, wws_points, hpc_points, hpt_points, esn)

In [ ]:
for i, sensor in enumerate(generals.SENSORS):
    sensor_name = sensor.value
    for esn in generals.ESN:
        fig = update_plot(sensor_name, train, esn)
        iplot(fig)

In [ ]:
esn101 = train[train['ESN'] == 101 ]
N = len(esn101) # Get the number of rows
esn101['i'] = pd.RangeIndex(start=0, stop=N)

esn102 = train[train['ESN'] == 102 ]
N = len(esn102) # Get the number of rows
esn102['i'] = pd.RangeIndex(start=0, stop=N)

esn103 = train[train['ESN'] == 103 ]
N = len(esn103) # Get the number of rows
esn103['i'] = pd.RangeIndex(start=0, stop=N)

esn104 = train[train['ESN'] == 104 ]
N = len(esn104) # Get the number of rows
esn104['i'] = pd.RangeIndex(start=0, stop=N)


train_2 = pd.concat([esn101, esn102, esn103, esn104], axis=0)